# Time series preprocessing

## Goal

This walkthrough creates a daily demand series with a missing date, a duplicate timestamp, and a missing value. It demonstrates validation, chronological repair, interpolation, lag features, and forecast-safe rolling features.

In [1]:
import numpy as np
import pandas as pd
from autoprepml import TimeSeriesPrepML

rng = np.random.default_rng(42)
dates = pd.date_range("2025-01-01", periods=35, freq="D").delete([7, 8])
demand = 100 + np.arange(len(dates)) * 1.5 + rng.normal(0, 2, len(dates))
frame = pd.DataFrame({"timestamp": dates, "demand": demand})
frame.loc[4, "demand"] = np.nan
frame = pd.concat([frame, frame.iloc[[3]]], ignore_index=True)
frame = frame.sample(frac=1, random_state=42).reset_index(drop=True)
frame.head()

,timestamp,demand
0,2025-01-18,120.781415
1,2025-01-22,128.400148
2,2025-01-30,141.230888
3,2025-01-29,140.064618
4,2025-01-11,111.966398


### Validate and repair the sequence

In [2]:
preparer = TimeSeriesPrepML(frame, timestamp_column="timestamp", value_column="demand")
before = preparer.detect_issues()
preparer.sort_by_time()
preparer.remove_duplicate_timestamps(aggregate="mean")
preparer.fill_missing_timestamps(freq="D")
preparer.interpolate_missing(method="linear")
after = preparer.detect_issues()
print("before:", {key: before[key] for key in ["duplicate_timestamps", "detected_gaps", "is_chronological"]})
print("after:", {key: after[key] for key in ["duplicate_timestamps", "detected_gaps", "is_chronological", "missing_values"]})

before: {'duplicate_timestamps': 1, 'detected_gaps': 1, 'is_chronological': False}
after: {'duplicate_timestamps': 0, 'detected_gaps': 0, 'is_chronological': True, 'missing_values': 0}


### Add features without leaking future values

The default rolling implementation shifts the input before calculating statistics. This keeps the current row from contributing to its own feature.

In [3]:
preparer.add_time_features()
preparer.add_lag_features(lags=[1, 7])
preparer.add_rolling_features(windows=[3, 7], functions=["mean", "std"])
columns = ["timestamp", "demand", "demand_lag_1", "demand_rolling_mean_3"]
preparer.df[columns].tail()

,timestamp,demand,demand_lag_1,demand_rolling_mean_3
30,2025-01-31,142.825465,141.230888,139.363746
31,2025-02-01,144.361642,142.825465,141.373657
32,2025-02-02,149.283295,144.361642,142.805998
33,2025-02-03,145.687170,149.283295,145.490134
34,2025-02-04,146.975515,145.687170,146.444036


## Checks

In [4]:
assert preparer.df["timestamp"].is_monotonic_increasing
assert preparer.df["timestamp"].duplicated().sum() == 0
assert preparer.df["demand"].isna().sum() == 0
assert "demand_rolling_mean_3" in preparer.df
print("Time series workflow checks passed.")

Time series workflow checks passed.
